## Paths and executables

In [1]:
conda activate ppanggolin
ppanggolin -v
conda deactivate

ppanggolin 2.0.5


In [2]:
# Input folders
genomesFolder="00-fastas/pan"

# Output folders
workFolder="01-pangenome"
mkdir -p $workFolder

# Root folder
root=$(pwd)

## Parameters

In [3]:
cluster_id=0.80
cluster_cov=0.60
n_cores=14
label="clostridiales"

## Construct pangenome

In [4]:
conda activate ppanggolin

Assuming you already have annotated your genomes using Bakta or obtained annotations from NCBI, put the GenBank proteome files (`*.gbff`) in some folder.

### Preventive cleanup of the annotations

In [5]:
find $genomesFolder/*.gbff | xargs -I % bash -c 'cat % | tr -d "’" > tmp && mv tmp %'

### Build!

In [6]:
paste <(dir -1 $genomesFolder | grep .gbff) <(find "$genomesFolder" | grep .gbff) > annotations.list

In [7]:
ppanggolin annotate -f --anno annotations.list -o $workFolder -c $n_cores

2025-10-06 13:18:01 utils.py:l168 INFO	Command: /home/lucas/bin/anaconda3/envs/ppanggolin/bin/ppanggolin annotate -f --anno annotations.list -o 01-pangenome -c 14
2025-10-06 13:18:01 utils.py:l169 INFO	PPanGGOLiN version: 2.0.5
2025-10-06 13:18:01 utils.py:l722 INFO	3 parameters have a non-default value.
2025-10-06 13:18:01 annotate.py:l503 INFO	Reading annotations.list the list of genome files ...
100%|███████████████████████████████████████| 209/209 [00:11<00:00, 18.26file/s]
2025-10-06 13:18:14 annotate.py:l535 INFO	gene identifiers used in the provided annotation files were unique, PPanGGOLiN will use them.
2025-10-06 13:18:14 writeBinaries.py:l709 INFO	Writing genome annotations...
100%|███████████████████████████| 19437/19437 [00:00<00:00, 966504.88genedata/s]
2025-10-06 13:18:19 writeBinaries.py:l720 INFO	writing the protein coding gene dna sequences in pangenome...
100%|█████████████████████████████| 781847/781847 [00:00<00:00, 800135.08gene/s]
2025-10-06 13:18:40 writeBinaries

In [8]:
ppanggolin cluster -f -p $workFolder/pangenome.h5 --identity $cluster_id --coverage $cluster_cov -c $n_cores \
--tmpdir /mnt/DATA/tmp

2025-10-06 13:18:45 utils.py:l168 INFO	Command: /home/lucas/bin/anaconda3/envs/ppanggolin/bin/ppanggolin cluster -f -p 01-pangenome/pangenome.h5 --identity 0.80 --coverage 0.60 -c 14 --tmpdir /mnt/DATA/tmp
2025-10-06 13:18:45 utils.py:l169 INFO	PPanGGOLiN version: 2.0.5
2025-10-06 13:18:45 utils.py:l722 INFO	4 parameters have a non-default value.
2025-10-06 13:18:45 readBinaries.py:l94 INFO	Getting the current pangenome status
2025-10-06 13:18:45 readBinaries.py:l236 INFO	Extracting and writing CDS sequences from a /mnt/DATA/PhD/WPs/WP1/clostridia_models_9090/01-pangenome/pangenome.h5 file to a fasta file...
100%|█████████████████████████████| 781847/781847 [00:04<00:00, 176643.62gene/s]
2025-10-06 13:19:09 cluster.py:l317 INFO	Clustering all of the genes sequences...
2025-10-06 13:19:09 cluster.py:l90 INFO	Creating sequence database...
2025-10-06 13:19:19 cluster.py:l97 INFO	Clustering sequences...
2025-10-06 13:29:39 cluster.py:l103 INFO	Extracting cluster representatives...
2025-10-

In [9]:
no_genomes=$(ppanggolin info -p $workFolder/pangenome.h5 --content | grep Genomes | grep -Eo '[0-9]+')

In [10]:
ppanggolin fasta -f -p $workFolder/pangenome.h5 -o $workFolder --prot_families softcore --soft_core $(echo "scale=5; 2/$no_genomes" | bc)

2025-10-06 13:54:51 utils.py:l168 INFO	Command: /home/lucas/bin/anaconda3/envs/ppanggolin/bin/ppanggolin fasta -f -p 01-pangenome/pangenome.h5 -o 01-pangenome --prot_families softcore --soft_core .00956
2025-10-06 13:54:51 utils.py:l169 INFO	PPanGGOLiN version: 2.0.5
2025-10-06 13:54:51 utils.py:l722 INFO	4 parameters have a non-default value.
2025-10-06 13:54:51 readBinaries.py:l94 INFO	Getting the current pangenome status
2025-10-06 13:54:51 readBinaries.py:l715 INFO	Reading pangenome annotations...
100%|███████████████████████████████| 19693/19693 [00:00<00:00, 226947.44gene/s]
2025-10-06 13:55:01 readBinaries.py:l730 INFO	Reading pangenome gene families...
100%|███████████████████████| 413540/413540 [00:05<00:00, 80937.43gene family/s]
2025-10-06 13:55:14 writeSequences.py:l109 INFO	Writing the representative amino acid sequences of the gene families in softcore genome, that are present in more than 0.00956 of genomes
100%|████████████████████| 114661/114661 [00:00<00:00, 295490.80

In [11]:
mv $workFolder/softcore_protein_families.faa $workFolder/$label.faa

In [12]:
conda deactivate